# Human-in-the-Loop: Ettevalmistusväravad, Riski tasustamine ja Auditilogimine

Selle õppetunni README-s tutvustatakse Human-in-the-Loop’i lühikese fragmendiga, mis küsib kasutajalt `APPROVE` või `REJECT` pärast seda, kui agent on juba vastuse andnud. See muster on hea lähtepunkt, kuid tootmises HITL rakendused vajavad tavaliselt kolme lisatükki:

1. **Ettevalmistusvärav**, mis käivitub **enne** agenti riskantse sammu täitmist, et hoida kulud, pöördumatus ja latentsus kontrolli all.
2. **Riskitasustamine**, nii et madala riskiga toimingud käivitatakse automaatselt, keskmise riskiga toimingud kinnitatakse hulgipõhiselt ja ainult kõrge riskiga toimingud ootavad inimese otsust.
3. **Auditilogigi pluss paranduse tsükkel**, nii et iga väravotsus salvestatakse JSONL-vormingus ja tagasilükkamine kutsutakse agenti üles struktureeritud põhjusega, mitte ainult „Revising...“ kuvamisega.

See märkmik ehitab iga neist üles samade primitiivide peal nagu `06-system-message-framework.ipynb`. See töötab läbi lõpuni `DEMO_MODE = True` (ei vaja interaktiivset sisendit) või päris `input()` päringutega, kui `DEMO_MODE = False`. Märkus: DEMO_MODE puhul on kolmanda eesmärgi uuesti proovimine skriptitud, nii et tsükli mehhanismid on lõpust lõpuni nähtavad. Tõeline paranduste-põhine uuesti klassifitseerimine nõuab `DEMO_MODE = False` ja operaatorit.

**Väljasjäänud ulatus (lahendatud teistes õppetundides):** autentimine ja ligipääsu kontroll (Õppetund 06 README oht 2), tööriista-kõnede vahendustarkvara (Õppetund 14 MAF süvauurimine), mitme agendi arutelu mustrid.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Muster 1: Eeltegevuse värav

README HITL lõik kutsub esmalt agendi välja ja seejärel palub kasutajal väljund heaks kiita. See on **järgtoimingu** voog. Agent on juba tegutsenud, seega on LLM-kõne kulu juba tasutud ning mis tahes kõrvalmõju (saadetud e-kiri, kirjutatud andmebaasirida, postitatud kommentaar) on juba toimunud.

**Eeltegevuse** voog lisab värava enne, kui agent riskantset sammu teeb. Agent teeb ettepaneku toiminguks, värav otsustab, kas käivitada, ning ainult heakskiidu korral toimub kõrvalmõju.

| Aspekt | Järgtoimingu heakskiit (README lõik) | Eeltegevuse värav (see märkmik) |
|---|---|---|
| Millal toimub heakskiit? | Pärast seda, kui agent on väljundi tootnud | Enne mis tahes kõrvalmõju teostamist |
| LLM kulu keeldumisel | Juba tasutud | Tasutud ainult ettepaneku eest, mitte toimingu eest |
| Tagasipöördumatud kõrvalmõjud | Võimalikud (toiming on juba toimunud) | Vältitud |
| Audit selgus | Heakskiit on prindilausena | Heakskiit on ajatempli, toimingu ja põhjusega JSON-kirje |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Muster 2: Riskiklassifikatsioon

Kõik toimingud ei vaja inimkinnitust. Avaliku API kaudu ainult lugemiseks tehtav päring on teistsuguse riskiga kui kliendile e-kirja saatmine. Mõlema sama kohtlemine raiskab operaatori tähelepanu ja aeglustab agendi tööd.

Lihtne 3-tasandiline mudel:

| Tase | Näited | Kinnitamise protsess |
|---|---|---|
| `madal` (ainult lugemine) | Otsi teadmistebaasist, vaata lennuvõimalusi, tõmba avalik veebileht | Automaatne täitmine, logitakse auditi jaoks |
| `keskmine` (odav muutmine) | Vahemälu tulemus, suurenda loendurit, aja meeldetuletus | Automaatne täitmine, kuid koondatud igapäevaseks ülevaatuseks |
| `kõrge` (väljapoole suunatud või pöördumatu) | Saada e-kiri, karda arvele võtta, postita avalikku kanalisse | Blokeeri inimkinnituse ootamisel |

See on üks klassifitseerimine. Tootmissüsteemides kasutatakse sageli detailsemaid tasandeid (nt AWS IAM õiguste tasemed, rollipõhised ligipääsu tasemed). Allpool olev 3-tasandiline versioon on väikseim kasulik versioon agendile, kes segab ainult lugemiseks mõeldud ja kõrvalmõjuga toiminguid.

Allolev klassifikaator kasutab märksõnade heuristikuid, et demo oleks deterministlik ja odav. Tootmissüsteemis asendaksite selle õpitud klassifikaatori või poliitikamootoriga.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Muster 3: Audiitoripäevik ja revisioonitsükkel

`print("Response approved.")` ei ole audiitoripäevik. Usalduse tagamiseks tuleks iga värava otsus salvestada struktureeritud sündmusena, mida saab hiljem pärida, taasesitada või kinnitada intsidenti ülevaatuseks.

Kaks osa:

1. **Lisamisega ainult JSONL.** Üks rida iga otsuse kohta koos ajahetke, tegevuse, taseme, otsuse ja põhjusega. Lihtne grep’ida, lihtne hiljem tõelisse logipoodi saata.
2. **Revisioonitsükkel tagasilükkamisel.** Kui värav tagastab `deny`, esitab agent endale tagasilükkamise põhjuse kontekstis uuesti, et järgmine ettepanek saaks vältida probleemi.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Lisamaterjalid

Mitmed teised avalikud projektid rakendavad nende HITL mustrite variatsioone. Võrdle lähenemisi, et leida, mis sobib sinu virnaga:

- **LangChain** inim-tsüklis tööriistade mähised ([dokumentatsioon](https://python.langchain.com/docs/integrations/tools/human_tools)): tööriistade mähised, mis peatavad täitmise inimsisendi jaoks.
- **AutoGen** `UserProxyAgent` ([v0.2 dokumentatsioon](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ muutis seda struktuuri): kasutab agendi rolli, mis esindab inimest mitme-agendi vestlustes.
- **Semantic Kernel** funktsioonide filtrid ([dokumentatsioon](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): vahendkihi filtrid, mis toimivad iga funktsiooni väljakutse ümber, sobivad loogika piiramiseks.

Iga projekt käsitleb kolme alakujundit erinevalt: LangChain mähib need tööriistadeks, AutoGen kasutab agendi rolli, Semantic Kernel kasutab vahendkihi filtreid. Loe üks või kaks rakendust algusest lõpuni läbi, enne kui valid oma agendi disaini.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
